# Step 2 — What does each column mean, and which values does it hold?

**Purpose.** Before asking questions we need to know every column: what it means and which values it really contains. This notebook prints, for each of the three raw tables, the data type, the number of missing and distinct values, and example values. For columns with few distinct values it prints all of them with their counts.

**Where the meanings come from.** The author only defines some columns. Every meaning below carries a tag:

| Tag | Meaning |
|---|---|
| **[A]** | The author states it in the Kaggle description or column descriptions |
| **[I]** | Inferred from the column name or from sample values, **not confirmed**. Check it against the output below |
| **[V]** | Verified by us from the real output of this notebook |

After running the notebook, change an **[I]** to **[V]** only when the printed values support it, and fix the meaning when they don't.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1 and 2 take seconds. Part 3 reads the 80M-row list table once and takes a few minutes; set `RUN_LISTS = False` in the setup cell to skip the full read.

**Where this step sits.** Step 2 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). It only looks at values and formats. Meanings that step 5 later confirms by statistics are re-confirmed on the cleaned data. Part 4 answers three questions that parts 1 to 3 left open.


## Data dictionary

### Anime table — `AnimeList.csv`, one row is an anime (31 columns)

| Column | Meaning | Tag |
|---|---|---|
| `anime_id` | MyAnimeList id of the anime, the key of this table | [A] |
| `title` | Title (romanized) | [A] |
| `title_english` | English title | [A] |
| `title_japanese` | Japanese title | [A] |
| `title_synonyms` | Other names of the anime | [A] |
| `image_url` | URL of the cover image | [A] |
| `type` | Format of the anime, for example TV or OVA | [I] |
| `source` | What the anime is based on, for example Manga or Original | [I] |
| `episodes` | Number of episodes (a 0 may mean unknown) | [I] |
| `status` | Airing status, for example Finished Airing | [I] |
| `airing` | Whether the anime is currently airing (True/False); checked row by row against `status` in part 1 | [I] |
| `aired_string` | Air dates as text, for example `Apr 6, 2018 to ?` | [I] |
| `aired` | Air dates in a second format, to be checked | [I] |
| `duration` | Length of one episode as text, for example `24 min. per ep.` | [I] |
| `rating` | Audience rating: a content label that implies a recommended age. It is not the users' score | [I] |
| `score` | Average score given by MAL users | [I] |
| `scored_by` | Number of users who scored it | [I] |
| `rank` | Rank by score. Stored as a decimal number but holds integers only, because of missing values | [A] |
| `popularity` | Rank by popularity | [I] |
| `members` | Number of users who have it on their list | [I] |
| `favorites` | Number of users who marked it as favorite | [I] |
| `background` | Free-text background information | [I] |
| `premiered` | Season it premiered in, for example Spring 2012 | [I] |
| `broadcast` | Weekday and time of broadcast | [I] |
| `related` | Related anime (sequel, prequel, adaptation) stored as text of a dictionary | [I] |
| `producer` | Producers, several in one cell | [I] |
| `licensor` | Licensors, several in one cell | [I] |
| `studio` | Animation studio(s) | [A] |
| `genre` | Genres, several in one cell | [A] |
| `opening_theme` | Opening songs | [I] |
| `ending_theme` | Ending songs | [I] |

### User table — `UserList.csv`, one row is a user (17 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user's name, used as the user key | [A] |
| `user_id` | Numeric id of the user | [I] |
| `user_watching` | Number of anime on the user's list with status watching | [I] |
| `user_completed` | ... with status completed | [I] |
| `user_onhold` | ... with status on hold | [I] |
| `user_dropped` | ... with status dropped | [I] |
| `user_plantowatch` | ... with status plan to watch | [I] |
| `user_days_spent_watching` | Total days the user spent watching | [I] |
| `gender` | Gender | [A] |
| `location` | Location, free text typed by the user | [A] |
| `birth_date` | Birth date | [A] |
| `access_rank` | Unknown, appears empty | [I] |
| `join_date` | Registration date | [A] |
| `last_online` | Last date the user was online | [A] |
| `stats_mean_score` | The user's mean score | [I] |
| `stats_rewatched` | Number of rewatched anime | [I] |
| `stats_episodes` | Total episodes watched | [I] |

The five `user_*` count columns are described by the author only as "aggregated values from their anime lists".

### List table — `UserAnimeList.csv`, one row is a user–anime pair (11 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user | [A] |
| `anime_id` | The anime | [A] |
| `my_watched_episodes` | Episodes the user has watched | [I] |
| `my_start_date` | Date the user started watching (`0000-00-00` may mean empty) | [I] |
| `my_finish_date` | Date the user finished | [I] |
| `my_score` | Score the user gave, 1–10. **0 means not rated**, since the 1–10 scale has no 0 | [A] |
| `my_status` | Status code: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch. **The file stores numbers, the words come from the description.** Other codes are unexplained | [A] |
| `my_rewatching` | Whether the user is rewatching | [I] |
| `my_rewatching_ep` | Episode of the rewatch | [I] |
| `my_last_updated` | Timestamp of the last update of this row | [A] |
| `my_tags` | Free-text tags written by the user | [I] |

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips the full read of the 80M-row list table
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))


def profile(df, n_examples=4):
    """One row per column: type, missing, distinct values and a few example values."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "distinct": int(s.nunique(dropna=True)),
            "examples": s.dropna().astype(str).str.slice(0, 45).unique()[:n_examples].tolist(),
        })
    return pd.DataFrame(rows).set_index("column")


def show_values(df, max_distinct=25):
    """Print every distinct value, with its count, for columns that have few distinct values."""
    for col in df.columns:
        distinct = df[col].nunique(dropna=False)
        if distinct <= max_distinct:
            counts = df[col].value_counts(dropna=False)
            try:
                counts = counts.sort_index()
            except TypeError:
                pass
            print(f"\n--- {col}: {distinct} distinct values")
            print(counts.to_string())

## 1. Anime table

For every column: its type, how many values are missing, how many distinct values it has, and a few examples. Read the `examples` column slowly and compare it with the dictionary above.

In [ ]:
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
print("shape:", anime.shape, "| anime_id unique:", anime.anime_id.is_unique)
print(profile(anime).to_string())

In [ ]:
# Every distinct value of the columns that have few of them
show_values(anime)

### NEW — `airing` against `status`, row by row (added after the first run)

**Why this check exists.** The dictionary says `airing` looks redundant with `status`. The first run only compared two **counts**: 351 anime have `airing = True` and 351 anime have `status = Currently Airing`. Equal counts do not prove that the two columns describe the same anime. They could be 351 different anime in each column.

**What the cell does.** It counts every combination of the two columns, row by row, and counts the anime where they disagree.

**How to read the result.**
- All 351 anime sit in the cell (`Currently Airing`, `True`) and the disagreement count is **0**: the two columns say the same thing on every anime, so `airing` is redundant and we can drop it later.
- Any other number: the columns disagree on some anime. We look at those anime before deciding, and the dictionary row of `airing` stays [I].


In [ ]:
# NEW: row-by-row check of airing against status (the first run compared only the two counts)
print(pd.crosstab(anime["status"], anime["airing"], margins=True))
disagree = anime["status"].eq("Currently Airing") != anime["airing"]
print("\nanime where airing disagrees with status = Currently Airing:", int(disagree.sum()))


### NEW — is the label "Not yet aired" believable? (added after the first run)

**Why this check exists.** `status = Not yet aired` is a label that MyAnimeList sets by hand and the crawl took months, so some of these anime may already have been airing. The label is relative to the crawl date (around 2018), not to today. We cannot say "336 anime have not aired", only "336 anime are **labelled** not yet aired". Other columns give independent evidence:

| Evidence | If the label is right | If the label is wrong |
|---|---|---|
| Start date inside `aired`, against a crawl date | Start date is later than the crawl date, or missing | Start date is already in the past |
| `scored_by` and `members` | Zero or very small | Large numbers |
| Rows of the list table with `my_status = 2` (completed), part 3 below | None or almost none | Many users "completed" it |

**The crawl date.** We do not know it exactly. A lower bound is the latest start date among the anime labelled `Currently Airing`: they were airing when the page was read, so the crawl cannot be earlier than their start date.

**How to read the result.** The count "start date on or before that bound" is the number of anime whose label looks stale. The ten earliest start dates are printed so we can look at the worst cases.


In [ ]:
# NEW: is the label "Not yet aired" believable? Evidence from other columns
import ast

def start_date(text):
    # `aired` is the text of a Python dictionary such as {'from': '2012-01-13T00:00:00+00:00', 'to': None}
    try:
        return pd.to_datetime(ast.literal_eval(text).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

anime["start"] = anime["aired"].map(start_date)
nya = anime[anime["status"] == "Not yet aired"]
bound = anime.loc[anime["status"] == "Currently Airing", "start"].max()

print("latest start date among Currently Airing anime (lower bound of the crawl date):", bound)
print("labelled Not yet aired:", len(nya))
print("  start date missing:", int(nya["start"].isna().sum()))
print("  start date later than the bound:", int((nya["start"] > bound).sum()))
print("  start date on or before the bound (label looks stale):", int((nya["start"] <= bound).sum()))
print("  scored_by > 0:", int((nya["scored_by"] > 0).sum()), "| members > 0:", int((nya["members"] > 0).sum()))
print()
print("the ten earliest start dates among them:")
print(nya.sort_values("start")[["anime_id", "title", "type", "start", "scored_by", "members"]].head(10).to_string(index=False))


### Observations for part 1

**TODO (fill in after the run):**
- For each **[I]** column of the anime dictionary: does the output confirm the meaning? Change the tag or correct the text.
- Columns with few distinct values: which values are there, and are any unexpected (typos, placeholders such as `Unknown`)?
- Columns with many distinct values: what format do the examples show (comma-separated lists, dictionaries as text, dates as text)?

## 2. User table

In [ ]:
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("shape:", users.shape, "| username unique:", users.username.is_unique)
print(profile(users).to_string())

In [ ]:
show_values(users)

print("\n--- location: top 15 of", users["location"].nunique(), "distinct values")
print(users["location"].value_counts().head(15).to_string())

### NEW — the rows that share a `user_id` (added after the first run)

**Why this check exists.** `username` is unique (302,675 different values) but `user_id` has only 302,673 different values, so some rows share an id with another row. We never looked at those rows. Two explanations are possible, and the output tells them apart:

- **The same account twice.** Everything except the username is equal (join date, last online, statistics, location), as if one account had been crawled under two names. Then one of the two rows is a duplicate.
- **Two different users with a wrong id.** Many columns differ. Then they are two real users and the id is simply not reliable.

**What the cell does.** It lists the shared rows with all their columns (one column per row, so nothing is cut), counts the rows that are identical in every column, and for each shared `user_id` prints the columns in which its rows differ.

**How to read the result.** If the rows differ only in `username`, they look like a duplicated account. If they differ in many columns, they are different users. Either way we keep joining on `username`, never on `user_id`.


In [ ]:
# NEW: the rows that share a user_id
dup_ids = users.loc[users["user_id"].duplicated(keep=False), "user_id"].unique()
shared = users[users["user_id"].isin(dup_ids)].sort_values("user_id")
print("user_id values used by more than one row:", len(dup_ids), "| rows involved:", len(shared))
print("rows identical in every column to another row:", int(users.duplicated(keep=False).sum()))
print()
print(shared.T.to_string())
print()
for uid, group in shared.groupby("user_id"):
    differ = [c for c in group.columns if group[c].nunique(dropna=False) > 1]
    print("user_id", uid, "-> the rows differ in:", differ)


### Observations for part 2

**TODO (fill in after the run):** the same questions as part 1 for the user table. Which gender values exist, how messy is `location`, what do `birth_date`, `join_date` and `last_online` look like, and is `access_rank` really empty?

## 3. List table

The file has about 80 million rows, so it is read in two ways:

1. **A sample of the first 1,000,000 rows, all columns as text.** This only shows the *formats* of the columns. The file is ordered by user, so the first million rows belong to a small group of users and must not be used for statistics.
2. **A full read of the columns that hold few distinct values or numbers** (one pass, a few minutes).

In [ ]:
sample = pd.read_csv(paths["UserAnimeList.csv"], nrows=1_000_000, dtype=str, **na_kwargs)
print("sample shape:", sample.shape, "| columns:", list(sample.columns))
print(profile(sample).to_string())
print()
print(sample.head(5).T.to_string())
del sample

In [ ]:
if RUN_LISTS:
    list_cols = {
        "my_status": "int8", "my_score": "int8", "my_rewatching": "category",
        "my_watched_episodes": "float32", "my_rewatching_ep": "float32",
        "my_start_date": "category", "my_finish_date": "category",
    }
    lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(list_cols), dtype=list_cols, **na_kwargs)
    print(f"rows: {len(lists):,}")

    status_names = {1: "watching", 2: "completed", 3: "on hold", 4: "dropped", 6: "plan to watch"}
    status_table = lists.my_status.value_counts().sort_index().to_frame("rows")
    status_table["meaning"] = status_table.index.map(status_names).fillna("unknown")
    status_table["share_pct"] = (status_table.rows / len(lists) * 100).round(3)
    print("\n--- my_status")
    print(status_table.to_string())

    print("\n--- my_score (0 = not rated)")
    print(lists.my_score.value_counts().sort_index().to_string())

    print("\n--- my_rewatching")
    print(lists.my_rewatching.value_counts(dropna=False).to_string())

    print("\n--- my_watched_episodes and my_rewatching_ep")
    print(lists[["my_watched_episodes", "my_rewatching_ep"]]
          .describe(percentiles=[.5, .99, .999]).round(1).to_string())

    for col in ["my_start_date", "my_finish_date"]:
        counts = lists[col].value_counts(dropna=False)
        print(f"\n--- {col}: {lists[col].nunique():,} distinct values | top 5:")
        print(counts.head(5).to_string())
        print("share of '0000-00-00':", f"{(lists[col] == '0000-00-00').mean():.1%}")
else:
    print("Skipped (RUN_LISTS = False)")

### NEW — do users have "completed" rows for anime labelled Not yet aired? (added after the first run)

This is the third piece of evidence of the table above and the strongest one: nobody can finish an anime that has not aired. The cell reads the list table in chunks, keeps only the rows of the anime labelled `Not yet aired`, and counts them by `my_status`. It also prints the latest `my_last_updated` of the whole file (a Unix time in seconds), which is a second view of the crawl date.

**How to read the result.** Many rows with status 2 (completed) for an anime mean that its label is stale. Rows with status 6 (plan to watch) are normal, because people can add an anime to their plans before it airs.


In [ ]:
# NEW: do users have completed rows for anime labelled Not yet aired?
if RUN_LISTS:
    nya_ids = set(anime.loc[anime["status"] == "Not yet aired", "anime_id"])
    parts, latest = [], 0
    reader = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["anime_id", "my_status", "my_last_updated"],
        dtype={"anime_id": "int32", "my_status": "int8", "my_last_updated": "float64"},
        chunksize=5_000_000, **na_kwargs)
    for chunk in reader:
        latest = max(latest, chunk["my_last_updated"].max())
        parts.append(chunk[chunk["anime_id"].isin(nya_ids)])
    rows = pd.concat(parts)

    print("latest my_last_updated in the whole file:", pd.to_datetime(latest, unit="s"))
    print("rows of the anime labelled Not yet aired:", f"{len(rows):,}",
          "| anime with at least one row:", rows["anime_id"].nunique(), "of", len(nya_ids))
    print()
    print(rows.groupby("my_status").size().rename("rows").to_string())
    completed = rows[rows["my_status"] == 2].groupby("anime_id").size().rename("completed_rows")
    print()
    print("anime with at least one completed row:", len(completed))
    top = completed.sort_values(ascending=False).head(10).reset_index().merge(
        anime[["anime_id", "title", "start"]], on="anime_id")
    print(top.to_string(index=False))


### Observations for part 3

**TODO (fill in after the run):** the formats of `my_last_updated` and `my_tags`, what `my_rewatching` holds, whether `0000-00-00` is common in the start and finish dates, and whether `my_watched_episodes` has impossible values. Then update the tags in the list-table dictionary.

## 4. Follow-up checks: three questions left open by parts 1 to 3

Running parts 1 to 3 again takes a long time, so this part holds only the code needed for three open questions. It reuses `anime` and `users`, which are loaded above, and it reads the 5 GB list table **once**, in chunks, keeping only a few thousand rows.

| # | Question | Why it matters |
|---|---|---|
| 1 | Do **both usernames of each shared-`user_id` pair** (found in part 2) have rows in the list table, and are those rows the same? | If both names own rows with the same anime, one person's list would be counted twice. |
| 2 | Are the **60 early start dates** of the anime labelled `Not yet aired` (found in part 1) real airing dates, or dates made up from a partial text? | It decides whether the label "Not yet aired" can be trusted for these 60 anime. |
| 3 | Are the users with an **empty profile** always the same 129 users, and do they own list rows? | We found 129 missing values in five profile columns. We assumed they are the same users. |

**How long it takes.** The anime and user checks take seconds. The pass over the list table reads chunks of 5 million rows and takes a few minutes. `RUN_LISTS = False` (first code cell of this notebook) skips it.


### Question 3 first (fast): the users with an empty profile

**What we found in step 2.** Five columns of the user table (`join_date`, `last_online`, `stats_mean_score`, `stats_rewatched`, `stats_episodes`) each miss exactly 129 values. We assumed these are the same 129 users, and that they are users whose profile page could not be read.

**What the cell does.** It checks that the five missing masks are identical, counts how many of these users declare anime in their list sizes (`user_watching`, ...), and checks whether the four usernames of the shared-`user_id` pairs are among them. Whether they own rows in the list table is answered in the second half of this part.


In [ ]:
miss_cols = ["join_date", "last_online", "stats_mean_score", "stats_rewatched", "stats_episodes"]
masks = {c: users[c].isna() for c in miss_cols}
print("missing values per column:", {c: int(m.sum()) for c, m in masks.items()})
print("the same users miss all five columns:", all(masks[c].equals(masks["join_date"]) for c in miss_cols))

no_profile = users[masks["join_date"]]
size_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
declared = no_profile[size_cols].sum(axis=1)
print("users with an empty profile:", len(no_profile))
print("  declare 0 anime:", int((declared == 0).sum()), "| declare more than 0:", int((declared > 0).sum()))
print("  declared list size: median", declared.median(), "| mean", round(declared.mean(), 1), "| max", declared.max())

dup_ids = users.loc[users["user_id"].duplicated(keep=False), "user_id"].unique()
pair_rows = users[users["user_id"].isin(dup_ids)].sort_values("user_id")
pair_names = set(pair_rows["username"])
print()
print("usernames of the shared-user_id pairs and whether their profile is empty:")
for _, r in pair_rows.iterrows():
    print(f"  user_id {r.user_id}: {r.username:15s} empty profile: {r.username in set(no_profile['username'])}")


### The one pass over the list table (questions 1 and 2 need it)

In one pass over the 80 million rows (chunks of 5 million) the cell keeps only:
- all rows of the four usernames of the shared-`user_id` pairs (question 1);
- the rows of the anime labelled `Not yet aired`, with `my_status` and `my_score` (question 2);
- which of the empty-profile usernames own at least one row (question 3).


In [ ]:
def start_date(text):
    # `aired` is the text of a Python dictionary such as {'from': '2012-01-13T00:00:00+00:00', 'to': None}
    try:
        return pd.to_datetime(ast.literal_eval(text).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

anime["start"] = anime["aired"].map(start_date)
nya = anime[anime["status"] == "Not yet aired"].copy()
bound = anime.loc[anime["status"] == "Currently Airing", "start"].max()
nya["group"] = "later or missing date"
nya.loc[nya["start"] <= bound, "group"] = "early date (on or before the bound)"
print("crawl bound (latest start date among Currently Airing):", bound)
print(nya["group"].value_counts().to_string())

if RUN_LISTS:
    nya_ids = set(nya["anime_id"])
    empty_profile_names = set(no_profile["username"])
    pair_parts, nya_parts, owners = [], [], set()
    reader = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "anime_id", "my_status", "my_score"],
        dtype={"anime_id": "int32", "my_status": "int8", "my_score": "int8"},
        chunksize=5_000_000, **na_kwargs)
    for chunk in reader:
        pair_parts.append(chunk[chunk["username"].isin(pair_names)])
        nya_parts.append(chunk.loc[chunk["anime_id"].isin(nya_ids), ["anime_id", "my_status", "my_score"]])
        owners |= set(chunk.loc[chunk["username"].isin(empty_profile_names), "username"].unique())
    pair_lists = pd.concat(pair_parts)
    nya_rows = pd.concat(nya_parts)
    print("rows kept: pairs", len(pair_lists), "| Not yet aired", len(nya_rows))


### Question 1: do both usernames of a pair own list rows, and are the rows the same?

**What the cell does.** For each shared `user_id` it prints how many list rows each of the two usernames owns, how many anime appear in both lists, and for how many of those anime the status and the score are also equal. It also prints the list size the profile declares, to compare with the rows.

**How to read the result.**
- **One name owns all the rows and the other owns none:** the list is counted once. The second name is just an empty duplicate of the user row, and we can drop it.
- **Both names own rows with mostly the same anime, status and score:** one person's list is stored twice, so we must keep only one name.
- **Both own rows with different anime:** they are two different lists, and the shared id is only an id error.


In [ ]:
if RUN_LISTS:
    for uid, group in pair_rows.groupby("user_id"):
        a, b = group["username"].tolist()
        ra, rb = pair_lists[pair_lists["username"] == a], pair_lists[pair_lists["username"] == b]
        declared_size = int(group[size_cols].sum(axis=1).iloc[0])
        print(f"user_id {uid}: declared list size {declared_size}")
        print(f"  {a}: {len(ra)} rows | {b}: {len(rb)} rows")
        both = ra.merge(rb, on="anime_id", suffixes=("_a", "_b"))
        same = (both["my_status_a"] == both["my_status_b"]) & (both["my_score_a"] == both["my_score_b"])
        print(f"  anime in both lists: {len(both)} | of them with the same status and score: {int(same.sum())}")


### Question 2: are the early start dates real airing dates?

**The idea.** Of the 336 anime labelled `Not yet aired`, 60 have a start date on or before the crawl bound. If those dates were **real airing dates**, people would already be watching and finishing these anime, so they would look **different** from the other 276: fewer plan-to-watch rows, more watching and completed rows, more raters. If the dates carry no information about airing (an expected date, or a date made up from a partial text), the two groups would look **the same**.

**What the cells do.** They compare the two groups on the share of plan-to-watch rows, the number of non-plan rows per anime, the number of anime with raters, and the months of the early dates. The last cell lists the early anime with the most non-plan rows.

**How to read the result.** Similar numbers in the two groups mean the early dates say nothing about airing. Clearly different numbers mean the label is stale for the early group.


In [ ]:
if RUN_LISTS:
    rows = nya_rows.merge(nya[["anime_id", "group"]], on="anime_id")
    by_status = rows.groupby(["group", "my_status"]).size().unstack(fill_value=0)
    by_status["rows"] = by_status.sum(axis=1)
    by_status["plan_to_watch_pct"] = (by_status[6] / by_status["rows"] * 100).round(2)
    print(by_status.to_string())

    non_plan = rows[rows["my_status"] != 6].groupby("anime_id").size().rename("non_plan_rows")
    nya_rows_info = nya.merge(non_plan, on="anime_id", how="left").fillna({"non_plan_rows": 0})
    summary = nya_rows_info.groupby("group").agg(
        anime=("anime_id", "size"),
        non_plan_rows_per_anime=("non_plan_rows", "mean"),
        anime_with_raters=("scored_by", lambda s: int((s > 0).sum())),
        median_members=("members", "median"))
    print()
    print(summary.round(2).to_string())

    early = nya_rows_info[nya_rows_info["group"].str.startswith("early")]
    print()
    print("months of the early start dates:")
    print(early["start"].dt.strftime("%Y-%m").value_counts().sort_index().to_string())
    print()
    print("early anime with the most non-plan rows:")
    print(early.sort_values("non_plan_rows", ascending=False)[
        ["anime_id", "title", "start", "scored_by", "members", "non_plan_rows"]].head(10).to_string(index=False))


### NEW — are the early start dates filled in? (added after the second run)

**Why this check exists.** Question 2 judges the early dates by what users do. The dates can also be tested directly. The text `aired_string` is what the page shows (`Oct, 2007 to Oct, 2008`), and `aired` holds the same information as a dictionary of two **complete** dates. When the text gives only a month or only a year, a complete date cannot be copied from it, so some part of the date has to be made up. If the early dates were made up in that way, they say nothing about airing, and the bunching in May 2018 would have a mechanical cause.

**What the cell does.**
1. For **all** anime, it sorts the text of the start into three kinds: a full date (`Apr 3, 1998`), a month and a year (`Jul, 2018`), or a year only (`2005`). For each kind it shows how many anime have a start date in `aired`, and the most frequent days of the month and months of those dates. If the dates of the partial kinds are spread like real dates, nothing is made up. If the same day numbers and the same month come back again and again, they are made up.
2. For the 60 early anime and for the 129 anime with a later date, it shows how the start is written in the text, and it lists the 60 early anime with their text and their date side by side, so that we can see a date that the text does not contain.

**How to read the result.** The early dates are **filled in** if their text is only a month or a year while `aired` has a complete date, and if the same days and the same month repeat. They are **real** if the text is a full date and matches.


In [ ]:
# NEW: are the early start dates filled in? The date in `aired` is compared with the text of `aired_string`
import re

def start_kind(t):
    first = str(t).split(" to ")[0].strip()
    if re.search(r"^[A-Z][a-z]{2} \d{1,2}, \d{4}$", first): return "full date"
    if re.search(r"^[A-Z][a-z]{2}, \d{4}$", first): return "month and year"
    if re.search(r"^\d{4}$", first): return "year only"
    return "other"

anime["start_kind"] = anime["aired_string"].map(start_kind)
print("all anime: how aired_string writes the start, and the start dates that `aired` holds for each kind")
for kind in ["full date", "month and year", "year only", "other"]:
    s = anime.loc[(anime["start_kind"] == kind) & anime["start"].notna(), "start"]
    n = int((anime["start_kind"] == kind).sum())
    line = f"  {kind:15s} anime: {n:>6,} | with a start date: {len(s):>6,}"
    if len(s):
        line += f" | day of the month, top 4: {s.dt.day.value_counts().head(4).to_dict()} | month, top 3: {s.dt.month.value_counts().head(3).to_dict()}"
    print(line)

early = nya[nya["group"].str.startswith("early")].copy()
early["start_kind"] = early["aired_string"].map(start_kind)
later = nya[nya["start"].notna() & ~nya["group"].str.startswith("early")].copy()
later["start_kind"] = later["aired_string"].map(start_kind)
print()
print(f"the {len(early)} early anime: how aired_string writes the start:", early["start_kind"].value_counts().to_dict())
print("  start dates on the 9th to the 12th of a month:", int(early["start"].dt.day.between(9, 12).sum()), "of", len(early))
print("  start dates in May 2018:", int((early["start"].dt.strftime("%Y-%m") == "2018-05").sum()))
print(f"the {len(later)} anime with a later start date: how aired_string writes the start:", later["start_kind"].value_counts().to_dict())
print()
print(early.sort_values("start")[["anime_id", "title", "aired_string", "start"]].to_string(index=False, max_colwidth=36))


### Question 3 (second half): do the empty-profile users own list rows?


In [ ]:
if RUN_LISTS:
    print("users with an empty profile:", len(empty_profile_names))
    print("  of them with at least one list row:", len(owners))
    print("  of them without any list row:", len(empty_profile_names - owners))
    declared_by_name = no_profile.set_index("username")[size_cols].sum(axis=1)
    no_rows = declared_by_name[list(empty_profile_names - owners)]
    print("  of those without rows, declaring more than 0 anime:", int((no_rows > 0).sum()))


### Observations for part 4

**TODO (fill in after the run):**
1. Question 3: are the five missing masks the same? How many of the 129 declare anime, and how many own rows? Are the two empty-profile names of the pairs among them?
2. Question 1: how many rows does each name own, and do the lists overlap?
3. Question 2: are the early group and the later group similar in the share of plan-to-watch rows, in the non-plan rows per anime and in raters? Are the early dates concentrated in a few months?
- Question 2, second look: how is the start written in `aired_string` for the 60 early anime (full date, month and year, year only)? Do the same days (9 to 12) and the month May come back in the dates of the partial texts, for all anime and for the early group? If yes, the early dates are filled in and say nothing about airing.
